Q1. Embedding a query

In [1]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)

v[0]

2026-06-29 21:58:24.064594357 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


-0.02058203437252893

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
documents[10]

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

Q2. Cosine similarity

In [5]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

content = ''

for doc in documents:
    if doc['filename'] == target_filename:
        content = doc['content']
        break

In [6]:
print(content[:500])

# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over ev


In [7]:
page_content_vector = embed.encode(content)

In [8]:
score = page_content_vector.dot(v)

In [9]:
score

0.36107026789538205

Q3. Chunking and search by hand

In [10]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
import numpy as np

In [18]:
X = np.array([embed.encode(chunk['content']) for chunk in chunks])  
scores = X.dot(v)
best_idx = scores.argmax()
print(chunks[best_idx]['content'][:500])

rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search takes a shortcut. Instead of
comparing against everything, it first narrows down to a region of
likely matches. Then it scores only within that region. It may miss the
absolute best match, but the results are still good and it's much
faster.

```text
NN (exact):    compare query against ALL documents ->


In [24]:
chunks[best_idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

Q4. Vector search with minsearch

In [20]:
from minsearch import VectorSearch

index = VectorSearch(keyword_fields=["filename"])
index.fit(X, chunks)

In [21]:
query = "What metric do we use to evaluate a search engine?"
v_query = embed.encode(query)

In [22]:
results = index.search(v_query, num_results=5)

In [23]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

Q5. Text search vs vector search

In [25]:
# 1. Vector search
query = "How do I store vectors in PostgreSQL?"
v_query = embed.encode(query)

vector_results = index.search(v_query, num_results=5)

In [26]:
vector_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [27]:
# 2. Text / keyword search
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

text_results = text_index.search(query, num_results=5)

In [28]:
vector_files = [r["filename"] for r in vector_results]
text_files = [r["filename"] for r in text_results]

vector_files, text_files

(['02-vector-search/lessons/08-pgvector.md',
  '02-vector-search/lessons/08-pgvector.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/08-pgvector.md',
  '02-vector-search/lessons/08-pgvector.md'],
 ['02-vector-search/lessons/02-embeddings.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/01-intro.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/01-intro.md'])

In [29]:
for file in vector_files:
    if file not in text_files:
        print(file)

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


Q6. Hybrid search

In [30]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [32]:
query6 = "How do I store vectors in PostgreSQL?"
query6_vecor = embed.encode(query6)
vector_results6 = index.search(query6_vecor, num_results=5)
text_results6 = text_index.search(query6, num_results=5)
results6 = rrf([vector_results6, text_results6], k=60, num_results=5)
print(results6[0]["filename"])

02-vector-search/lessons/08-pgvector.md
